# Ejercicio 2: el 95% que no detecta nada

`pagos_taxi.csv`: 6000 pagos de la app de taxis, de septiembre y octubre de 2026. `es_fraude` vale 1 si el pago resultó ser fraudulento (tarjeta robada o contracargo). Mismo esqueleto del Ejercicio 1: `evaluar` corre una validación cruzada con el esquema que le pases y guarda una fila por modelo en `resultados`.

Los pasos 1 a 3 tienen `TODO`; los pasos 4 y 5 solo dicen qué variable dejar. La última celda imprime una comprobación por paso; no la modifiques. Corre todas las celdas una vez antes de empezar (**Run All**).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pagos = pd.read_csv("pagos_taxi.csv")
pagos["hora"] = pd.to_datetime(pagos["fecha_hora"]).dt.hour
pagos["monto_por_km"] = pagos["monto"] / pagos["distancia_km"]

NUMERICAS = ["monto", "distancia_km", "monto_por_km", "hora", "tarjeta_nueva", "intentos_fallidos", "viajes_previos"]
CATEGORICAS = ["metodo"]
X = pagos[NUMERICAS + CATEGORICAS]
y = pagos["es_fraude"]
print("Pagos:", len(pagos), " fraudes:", y.sum(), f" ({y.mean():.1%})")

## La función que evalúa cada modelo

In [ ]:
resultados = {}


def preparacion():
    return ColumnTransformer([
        ("numericas", StandardScaler(), NUMERICAS),
        ("categoricas", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAS),
    ])


def evaluar(nombre, modelo, esquema):
    """Validacion cruzada del modelo con el esquema dado. Guarda en
    resultados el promedio de cada metrica entre los pliegues."""
    pipeline = Pipeline([("preparar", preparacion()), ("modelo", modelo)])
    puntajes = cross_validate(pipeline, X, y, cv=esquema, scoring=["accuracy", "recall", "precision"])
    resultados[nombre] = {
        "modelo": nombre,
        "accuracy": round(puntajes["test_accuracy"].mean(), 4),
        "recall": round(puntajes["test_recall"].mean(), 4),
        "precision": round(puntajes["test_precision"].mean(), 4),
    }
    print(f"{nombre:20s} accuracy {resultados[nombre]['accuracy']:.4f}   recall {resultados[nombre]['recall']:.4f}   precision {resultados[nombre]['precision']:.4f}")

## Paso 1: un esquema que respete la proporción de fraudes

In [ ]:
# TODO 1: el esquema de validacion. Con 5% de fraudes, cada pliegue tiene
# que tener la misma proporcion de fraudes que el total: usa el esquema
# estratificado de 5 pliegues, mezclado, con random_state=42.
esquema = KFold(n_splits=5, shuffle=True, random_state=42)

for numero, (_, indices_validacion) in enumerate(esquema.split(X, y)):
    print(f"Pliegue {numero + 1}: {y.iloc[indices_validacion].sum()} fraudes de {len(indices_validacion)}")

## Paso 2: la línea base

In [ ]:
# TODO 2: la linea base, un clasificador que siempre predice la clase mas
# frecuente, con el nombre "linea base".

## Paso 3: dos modelos

In [ ]:
# TODO 3: una regresion logistica (max_iter=1000) con el nombre
# "logistica" y un gradient boosting (random_state=42) con el nombre
# "gradient boosting".

## Paso 4: la matriz de confusión

In [ ]:
# Deja en probabilidades la probabilidad de fraude que el gradient boosting
# le asigna a cada pago cuando ese pago esta en el pliegue de validacion
# (cada pago recibe una sola, del modelo que no lo vio al entrenar), y en
# matriz la matriz de confusion de esas probabilidades con el umbral 0.5.
probabilidades = None
matriz = None

## Paso 5: el umbral según el costo de cada error

In [ ]:
# Cada fraude que pasa sin detectar le cuesta a la empresa 40 dolares (el
# contracargo, en promedio). Cada pago legitimo que se marca como fraude le
# cuesta 3 (la revision manual y el pasajero molesto). Calcula el costo
# total para cada umbral de UMBRALES, guardalo en costos (un diccionario
# umbral -> costo) y deja en umbral_elegido el de menor costo.
COSTO_FRAUDE_NO_DETECTADO = 40
COSTO_FALSA_ALARMA = 3
UMBRALES = [0.05, 0.1, 0.15, 0.2, 0.3, 0.5]
costos = {}
umbral_elegido = None

## Comprobación

In [ ]:
def comprobar(numero, condicion):
    if condicion:
        print(f"Paso {numero}: OK")
    else:
        print(f"Paso {numero}: revisa")


def fila(nombre):
    return resultados.get(nombre)


comprobar(1, isinstance(esquema, StratifiedKFold) and esquema.shuffle and esquema.random_state == 42)
comprobar(2, fila("linea base") is not None and fila("linea base")["accuracy"] > 0.95 and fila("linea base")["recall"] == 0)
comprobar(3, fila("logistica") is not None and fila("gradient boosting") is not None and fila("gradient boosting")["recall"] > fila("logistica")["recall"])
comprobar(4, matriz is not None and matriz.sum() == len(pagos) and probabilidades is not None and len(probabilidades) == len(pagos))
comprobar(5, len(costos) == len(UMBRALES) and umbral_elegido == 0.1)

## Si terminas antes

La línea base tiene más accuracy que una moneda al aire y no detecta ni un fraude. ¿Qué número de la tabla de `resultados` lo delata, y por qué la accuracy no puede?